# 02 - Jetson Orin Nano (JetPack 6.2.3): setup and health check  (run on the Jetson)


In [2]:

import platform, subprocess, shutil, sys
print("python   :", sys.version.split()[0], "(system Python on JetPack 6 is 3.10)")
print("machine  :", platform.machine(), "| glibc:", platform.libc_ver())
for cmd in ["cat /etc/nv_tegra_release", "free -h", "df -h / | tail -1", "nproc"]:
    print("$", cmd); print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip(), "\n")
print("tegrastats:", shutil.which("tegrastats"), "| jtop:", shutil.which("jtop"))

python   : 3.10.12 (system Python on JetPack 6 is 3.10)
machine  : aarch64 | glibc: ('glibc', '2.35')
$ cat /etc/nv_tegra_release
# R36 (release), REVISION: 5.2, GCID: 46426093, BOARD: generic, EABI: aarch64, DATE: Thu Jul 16 18:56:22 UTC 2026
# KERNEL_VARIANT: oot
TARGET_USERSPACE_LIB_DIR=nvidia
TARGET_USERSPACE_LIB_DIR_PATH=usr/lib/aarch64-linux-gnu/nvidia 

$ free -h
total        used        free      shared  buff/cache   available
Mem:           7.4Gi       2.1Gi       156Mi        67Mi       5.2Gi       5.0Gi
Swap:          3.7Gi       2.0Mi       3.7Gi 

$ df -h / | tail -1
/dev/mmcblk0p1  113G   38G   70G  36% / 

$ nproc
6 

tegrastats: /usr/bin/tegrastats | jtop: None


## 2) Power mode (run these in a **terminal**, they need `sudo`)


## 3) Python packages for vision + pipeline + tracker (system Python 3.10)


## 4) Build the Llama runner (one time, in a terminal, with internet)


In [ ]:
%%writefile ../scripts/build_llama_runner_jetson.sh
#!/usr/bin/env bash
# Build the ExecuTorch Llama runner (`llama_main`) natively on a Jetson Orin Nano Developer Kit 8GB
# (aarch64, JetPack 6.2.x = Ubuntu 22.04, system Python 3.10, gcc 11, 128 GB SD card).
# - Needs internet ONCE (git, pip, apt). The finished binary then runs fully offline.
# - Takes several hours on an Orin Nano with 8 GB RAM and an SD card (the build is deliberately limited to a few
#   parallel jobs so it does not run out of memory). Run it inside `tmux` so closing the terminal does not kill it.
# - ET_BRANCH must match the executorch version in llama_deploy/export_info.json (notebook 01): yours is 1.6.0.
# - Written from the ExecuTorch docs; it could not be tested on an Orin Nano before delivery. If a step fails, read the
#   error, fix that step, and re-run: the script is safe to re-run.
set -euo pipefail

ET_BRANCH="${ET_BRANCH:-release/1.6}"
ET_TAG="${ET_TAG:-v1.6.0}"                           # used if the branch name does not exist
WORK="${WORK:-$HOME/et_build}"
REPO_DIR="${REPO_DIR:-$HOME/steel_defect_edge}"      # where this repo lives (the binary is copied to $REPO_DIR/bin)
JOBS="${JOBS:-3}"                                    # parallel compile jobs; 8 GB RAM is tight, raise only if the build is stable
SWAP_GB="${SWAP_GB:-8}"
mkdir -p "$WORK" "$REPO_DIR/bin"
cd "$WORK"
export CMAKE_BUILD_PARALLEL_LEVEL="$JOBS" MAX_JOBS="$JOBS"

echo "== 0) disk and swap =="
df -h / | tail -1
if ! swapon --show | grep -q /swapfile; then
  # Jetson images already have small zram swap; the build needs more. A swap file on the SD card is slow and wears
  # the card a little, so it is removed at the end of this script (see last lines).
  sudo fallocate -l "${SWAP_GB}G" /swapfile && sudo chmod 600 /swapfile && sudo mkswap /swapfile && sudo swapon /swapfile
fi

echo "== 1) system packages =="
sudo apt-get update
sudo apt-get install -y build-essential git curl wget pkg-config libssl-dev python3-pip python3-dev tmux

echo "== 2) Python 3.11 environment (JetPack's Python 3.10 is too old for ExecuTorch 1.6) =="
if [ ! -d "$HOME/miniforge3" ]; then
  curl -L -o miniforge.sh https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-aarch64.sh
  bash miniforge.sh -b -p "$HOME/miniforge3"
fi
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda create -y -n et311 python=3.11 || true
conda activate et311
pip install --upgrade pip cmake ninja     # Ubuntu 22.04's apt cmake (3.22) is older than ExecuTorch needs

echo "== 3) ExecuTorch source ($ET_BRANCH, fallback tag $ET_TAG) =="
if [ ! -d executorch ]; then
  git clone --branch "$ET_BRANCH" https://github.com/pytorch/executorch.git \
    || git clone --branch "$ET_TAG" https://github.com/pytorch/executorch.git
fi
cd executorch
git submodule sync && git submodule update --init --recursive
./install_executorch.sh

echo "== 4) build the Llama runner (CPU / XNNPACK) =="
make llama-cpu
# If `make llama-cpu` does not exist on your branch, use the workflow presets instead:
#   cmake --workflow --preset llm-release
#   cd examples/models/llama && cmake --workflow --preset llama-cpu

BIN=cmake-out/examples/models/llama/llama_main
ls -lh "$BIN"
cp "$BIN" "$REPO_DIR/bin/llama_main"
find cmake-out -name 'libllama_runner*.so*' -exec cp {} "$REPO_DIR/bin/" \; || true

echo "== 5) remove the temporary swap file (saves SD card space and wear) =="
sudo swapoff /swapfile 2>/dev/null && sudo rm -f /swapfile || true

echo "Done. Runner copied to $REPO_DIR/bin/llama_main"
echo "Next: copy llama_deploy/ (the .pte and tokenizer.model) into $REPO_DIR, then run notebook 02's checker."

## 5) Settings file and health check


In [3]:
%%writefile ../edge_inspect/settings.py
"""settings.py - one place for file locations. Edit settings.json (created next to the notebooks' parent folder)."""
from __future__ import annotations

import json
from pathlib import Path

DEFAULTS = {
    "onnx": "deploy/yolov8n_neu_combined.onnx",
    "meta": "deploy/model_meta.json",
    "demo_images": "deploy/demo_images",
    "severity_config": "severity_config.json",
    "llama_main": "bin/llama_main",
    "llama_model": "llama_deploy/llama32_1b_instruct_8da4w.pte",
    "llama_tokenizer": "llama_deploy/tokenizer.model",
    "runs_dir": "runs",
    "tracker_url": "http://127.0.0.1:5000",
}


def repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd.parent if cwd.name == "notebooks" else cwd


def load(root: Path = None) -> dict:
    root = Path(root) if root else repo_root()
    cfg = dict(DEFAULTS)
    f = root / "settings.json"
    if f.exists():
        cfg.update(json.loads(f.read_text()))
    out = {}
    for k, v in cfg.items():
        if k == "tracker_url":
            out[k] = v
        else:
            p = Path(v).expanduser()
            out[k] = str(p if p.is_absolute() else root / p)
    out["root"] = str(root)
    return out

Overwriting ../edge_inspect/settings.py


In [4]:

import json, sys
from pathlib import Path
REPO = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(REPO))
from edge_inspect import settings
if not (REPO / "settings.json").exists():
    (REPO / "settings.json").write_text(json.dumps(settings.DEFAULTS, indent=2))
S = settings.load(REPO)
for k, v in S.items(): print(f"{k:16s} {v}")

onnx             /home/armai_soc/steel_defect_edge/deploy/yolov8n_neu_combined.onnx
meta             /home/armai_soc/steel_defect_edge/deploy/model_meta.json
demo_images      /home/armai_soc/steel_defect_edge/deploy/demo_images
severity_config  /home/armai_soc/steel_defect_edge/severity_config.json
llama_main       /home/armai_soc/steel_defect_edge/bin/llama_main
llama_model      /home/armai_soc/steel_defect_edge/llama_deploy/llama32_1b_instruct_8da4w.pte
llama_tokenizer  /home/armai_soc/steel_defect_edge/llama_deploy/tokenizer.model
runs_dir         /home/armai_soc/steel_defect_edge/runs
tracker_url      http://127.0.0.1:5000
root             /home/armai_soc/steel_defect_edge


In [5]:

import os, re, subprocess, time, importlib

def row(name, ok, detail="", info=False):
    print(f"{'PASS' if ok else ('INFO' if info else 'FAIL')}  {name:34s} {detail}")
    return ok

results = {}
results["python"] = row("python version", sys.version_info >= (3, 8), sys.version.split()[0])
for mod in ["numpy", "cv2", "onnxruntime", "flask", "psutil"]:
    try:
        m = importlib.import_module(mod); results[mod] = row("import " + mod, True, getattr(m, "__version__", ""))
    except Exception as e:
        results[mod] = row("import " + mod, False, str(e)[:80])
try:
    import onnxruntime as ort
    prov = ort.get_available_providers()
    row("onnxruntime GPU provider", "CUDAExecutionProvider" in prov, str(prov) + "  (optional: CPU still works, just slower)", info=True)
except Exception:
    pass

for key in ["onnx", "meta", "llama_model", "llama_tokenizer", "llama_main"]:
    p = Path(S[key]); results[key] = row(f"file: {key}", p.exists(), (f"{p.stat().st_size/1e6:.1f} MB" if p.exists() else str(p)))
imgs = list(Path(S["demo_images"]).glob("*")) if Path(S["demo_images"]).exists() else []
results["demo_images"] = row("demo images", len(imgs) > 0, f"{len(imgs)} files")
row("tegrastats (power/RAM log)", shutil.which("tegrastats") is not None, "needed for the power numbers on the Jetson", info=True)


if results.get("llama_main") and results.get("llama_model") and results.get("llama_tokenizer"):
    env = dict(os.environ); env["LD_LIBRARY_PATH"] = str(Path(S["llama_main"]).parent) + ":" + env.get("LD_LIBRARY_PATH", "")
    help_txt = subprocess.run([S["llama_main"], "--help"], capture_output=True, text=True, env=env, timeout=60)
    row("llama_main --help runs", "model_path" in (help_txt.stdout + help_txt.stderr), "")
    prompt = ("<|start_header_id|>user<|end_header_id|>\n\nName one steel surface defect in 5 words.<|eot_id|>"
              "<|start_header_id|>assistant<|end_header_id|>\n\n")
    t0 = time.time()
    r = subprocess.run([S["llama_main"], f"--model_path={S['llama_model']}", f"--tokenizer_path={S['llama_tokenizer']}",
                        f"--prompt={prompt}", "--seq_len=80", "--temperature=0.2"],
                       capture_output=True, text=True, env=env, timeout=600)
    ok = r.returncode == 0 and len(r.stdout.strip()) > 0
    row("llama smoke test", ok, f"{time.time()-t0:.1f}s, return code {r.returncode}")
    print("---- stdout ----\n", r.stdout[-500:], "\n---- stderr tail ----\n", r.stderr[-600:])
else:
    print("Llama files not complete yet - finish Section 4 and copy llama_deploy/ first.")

PASS  python version                     3.10.12
PASS  import numpy                       1.26.4
PASS  import cv2                         4.8.0


2026-11-21 10:49:04.007784829 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card1/device/vendor"


PASS  import onnxruntime                 1.24.0


/tmp/ipykernel_18189/4173616413.py:12: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.1. Use feature detection or 'importlib.metadata.version("flask")' instead.
  m = importlib.import_module(mod); results[mod] = row("import " + mod, True, getattr(m, "__version__", ""))


PASS  import flask                       3.0.3
PASS  import psutil                      7.2.2
PASS  onnxruntime GPU provider           ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']  (optional: CPU still works, just slower)
PASS  file: onnx                         12.7 MB
PASS  file: meta                         0.0 MB
PASS  file: llama_model                  860.4 MB
PASS  file: llama_tokenizer              2.2 MB
PASS  file: llama_main                   6.3 MB
PASS  demo images                        29 files
PASS  tegrastats (power/RAM log)         needed for the power numbers on the Jetson
PASS  llama_main --help runs             
PASS  llama smoke test                   4.4s, return code 0
---- stdout ----
 {"prefill_token_per_sec":72.2433,"decode_token_per_sec":23.3333,"prompt_tokens":19,"generated_tokens":7,"model_load_start_ms":1795238344638,"model_load_end_ms":1795238348118,"inference_start_ms":1795238348118,"inference_end_ms":1795238348681,"mod

### If something fails
